# Lakebase & Vector Search Permission Setup

This notebook grants the necessary permissions to the app's service principal:

* **Cell 1**: Grants Lakebase database table permissions (SELECT, INSERT, UPDATE, DELETE)
* **Cell 2**: Grants Vector Search endpoint permissions (CAN_USE)

Run these cells **once** after creating the app to enable database and vector search access.

**Service Principal ID**: `e372d7cb-6577-482d-a925-2913debc45c2`

In [0]:
from databricks.sdk import WorkspaceClient
import psycopg2

SP_ID = "e372d7cb-6577-482d-a925-2913debc45c2"
ENDPOINT_NAME = "projects/jobcopilot-db-yourname/branches/production/endpoints/primary"

w = WorkspaceClient()

# Get host + token as YOUR identity (screengu@gmail.com)
endpoint = w.postgres.get_endpoint(name=ENDPOINT_NAME)
host = endpoint.status.hosts.host
me = w.current_user.me().user_name
token = w.postgres.generate_database_credential(endpoint=ENDPOINT_NAME).token
print(f"Connected as: {me}")
print(f"Host: {host}")

# Connect to Postgres as your identity
conn = psycopg2.connect(
    host=host,
    port=5432,
    dbname="databricks_postgres",
    user=me,
    password=token,
    sslmode="require"
)
conn.autocommit = True
cur = conn.cursor()

# Discover all tables and sequences in public schema
cur.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY tablename")
tables = [r[0] for r in cur.fetchall()]

cur.execute("SELECT sequence_name FROM information_schema.sequences WHERE sequence_schema = 'public'")
sequences = [r[0] for r in cur.fetchall()]

print(f"Tables: {tables}")
print(f"Sequences: {sequences}")

# Grant DML permissions on all existing tables to the app's service principal
cur.execute(f'GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA public TO "{SP_ID}"')
print(f"  Granted SELECT, INSERT, UPDATE, DELETE on {len(tables)} tables")

# Grant sequence permissions (needed for auto-increment columns)
cur.execute(f'GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO "{SP_ID}"')
print(f"  Granted USAGE, SELECT on {len(sequences)} sequences")

# Grant schema-level permissions
cur.execute(f'GRANT USAGE, CREATE ON SCHEMA public TO "{SP_ID}"')
print("  Schema public -> USAGE, CREATE granted")

# Default privileges so future tables you create are also accessible to the SP
cur.execute(f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO "{SP_ID}"')
cur.execute(f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT USAGE, SELECT ON SEQUENCES TO "{SP_ID}"')
print("  Default privileges set for future objects")

conn.close()
print("\nDone — all ownership transferred to service principal.")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

w = WorkspaceClient()

ENDPOINT_ID = "173f7718-b3d3-4d09-b22c-dc8bd87caec8"
SP_ID = "e372d7cb-6577-482d-a925-2913debc45c2"

w.permissions.update(
    request_object_type="vector-search-endpoints",
    request_object_id=ENDPOINT_ID,
    access_control_list=[
        AccessControlRequest(
            service_principal_name=SP_ID,
            permission_level=PermissionLevel.CAN_USE
        )
    ]
)
print(f"Granted CAN_USE on vector search endpoint to SP {SP_ID}")

# Verify
perms = w.permissions.get(request_object_type="vector-search-endpoints", request_object_id=ENDPOINT_ID)
for acl in perms.access_control_list:
    principal = acl.user_name or acl.group_name or acl.service_principal_name or "unknown"
    levels = [p.permission_level for p in acl.all_permissions]
    print(f"  {principal}: {levels}")